# Part 1 — 양자화(Quantization) 실습 (EXAONE 4.0 1.2B)
### EXAONE 4.0을 RTX 3080에 올리기 위한 첫 단계

> **환경**: 윈도우 로컬 VSCode + venv (`Python 3.11 (finetuning)` 커널) + RTX 3080  
> **최종 목표**: EXAONE 4.0 1.2B 법률 Q&A 파인튜닝 (Part 6까지 이어짐)  
> **이번 파트 목표**: 모델이 **4.8GB → 1GB**로 줄어드는 과정을 직접 확인

---

## 🎯 학습 목표

| # | 개념 | 결과물 |
|---|------|--------|
| 1 | 모델 크기 계산 공식 | FP32/FP16/4-bit 메모리 직접 계산 |
| 2 | FP16으로 EXAONE 4.0 로드 | VRAM ~2.4GB 확인 |
| 3 | 4-bit NF4로 EXAONE 4.0 로드 | VRAM ~1GB 확인 |
| 4 | `compute_dtype=bfloat16` 사용 | EXAONE 4.0 공식 권장값 |
| 5 | 양자화 전후 응답 품질 비교 | 품질 유지 체감 |

**실행 환경 확인:**  
우측 상단 커널이 **`Python 3.11 (finetuning)`** 인지 먼저 확인하세요.

---
## 0️⃣ 환경 준비

**이론 요약**
- EXAONE 4.0은 `transformers >= 4.54.0` 필요
- 양자화 핵심 라이브러리: **`bitsandbytes`** (Windows 0.43+ 지원)
- GPU 메모리를 **반복 측정**할 것이므로 함수로 준비

### 미션 0-1. 환경 검증

In [ ]:
import sys
import torch
import transformers

print(f"Python:       {sys.executable}")
print(f"PyTorch:      {torch.__version__}")
print(f"transformers: {transformers.__version__}  (>=4.54.0 필요)")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ **결과 확인**: 
- Python 경로에 **`venv`**가 포함되고, GPU 이름이 **RTX 3080**이면 OK
- **transformers 4.54 이상**이어야 EXAONE 4.0 사용 가능

👉 `AppData` 경로면 커널을 `Python 3.11 (finetuning)`로 바꾸세요.  
👉 transformers 4.54 미만이면 `%pip install --upgrade transformers>=4.54.0` 실행

### 미션 0-2. VRAM 확인 함수

In [ ]:
import gc

def print_gpu_memory(라벨=""):
    사용 = torch.cuda.memory_allocated() / 1e9
    전체 = torch.cuda.get_device_properties(0).total_memory / 1e9
    바 = "█" * int(사용 / 전체 * 30)
    빈 = "░" * (30 - len(바))
    print(f"[{라벨}] {사용:.2f}GB / {전체:.1f}GB  [{바}{빈}] {사용/전체*100:.0f}%")

def 메모리_해제():
    gc.collect()
    torch.cuda.empty_cache()

print_gpu_memory("초기 상태")

---
## 1️⃣ 모델 크기 계산 — 공식부터

**이론 요약**
- 공식: `메모리(bytes) = 파라미터 수 × 1개당 바이트 수`
- FP32=4bytes, FP16=2bytes, INT8=1byte, **4-bit=0.5byte**
- EXAONE 4.0 1.2B = **12억 파라미터**

💡 **왜 중요한가?**  
강의에서 본 숫자들을 **직접 계산**하면 "왜 4.8GB인가", "왜 1GB인가"가 즉시 이해됩니다.

### 미션 1-1. 정밀도별 메모리 계산

In [ ]:
파라미터_수 = 1_200_000_000  # EXAONE 4.0 1.2B

정밀도별_바이트 = {"FP32": 4, "FP16": 2, "INT8": 1, "4-bit": 0.5}

print(f"📊 EXAONE 4.0 1.2B ({파라미터_수/1e9:.1f}B 파라미터)")
print()
for 이름, 바이트 in 정밀도별_바이트.items():
    메모리 = 파라미터_수 * 바이트 / 1e9
    print(f"{이름:8s} ({바이트:.1f} bytes): {메모리:.2f} GB")

✅ **결과 확인**: FP32=4.80, FP16=2.40, 4-bit=0.60 GB 나왔나요?  
👉 **RTX 3080 10GB에는 모든 정밀도가 로드 가능!** 학습 메모리 여유도 충분.

### 미션 1-2. RTX 3080에서 학습까지 가능한지 체크

In [ ]:
VRAM = 10  # RTX 3080 10GB  132222

print(f"{'구성':30s} {'모델':>7s} {'학습추가':>9s} {'합계':>7s} {'가능?':>6s}")
print("-" * 65)

구성들 = [
    ("FP32 + 전체 학습", 4.8, 15),
    ("FP16 + 전체 학습", 2.4,  7),
    ("4-bit + LoRA",     1.0,  3),
]
for 이름, 모델, 추가 in 구성들:
    합계 = 모델 + 추가
    가능 = "✅" if 합계 < VRAM else "❌"
    print(f"{이름:30s} {모델:>6.1f}G {추가:>8.1f}G {합계:>6.1f}G {가능:>6s}")

✅ **결과 확인**: `4-bit + LoRA`가 ✅ 나왔나요?  
👉 이것이 우리가 Part 3에서 만들 **QLoRA** 구성입니다.  
👉 EXAONE 4.0 1.2B는 FP16도 가능하지만, 여유로운 학습을 위해 4-bit 권장.

---
## 2️⃣ FP16으로 EXAONE 4.0 로드 — 기준선 측정

**이론 요약**
- 먼저 **양자화 없이** 로드해서 기준선 VRAM 측정
- FP16 = 파라미터당 2 bytes → 이론값 2.4GB 예상
- 첫 다운로드는 ~2.5GB, 2~3분 소요 (이후 캐시됨)

💡 **왜 중요한가?**  
4-bit 양자화의 효과를 **FP16 기준선과 비교**해야 체감됩니다.

### 미션 2-1. HuggingFace 로그인

> ⚠️ **사전 준비**:  
> 1. `.env`에 `HUGGINGFACE_TOKEN` 저장  
> 2. HF 웹에서 [EXAONE-4.0-1.2B](https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B) Access Request 승인

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))
print("✅ HuggingFace 로그인 완료")

### 미션 2-2. 토크나이저 로드

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"

# ✨ EXAONE 4.0은 trust_remote_code 불필요 (HF 공식 지원)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"✅ 토크나이저 로드 완료 (어휘: {tokenizer.vocab_size:,})")

### 미션 2-3. FP16으로 모델 로드

> ⏱️ 첫 실행 시 ~2.5GB 다운로드로 **2~3분** 소요

In [ ]:
from transformers import AutoModelForCausalLM

print("🔄 FP16 로드 중... (첫 실행은 2~3분)")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print_gpu_memory("FP16 로드 후")

✅ **결과 확인**: 약 **2.4GB** 사용되나요?  
👉 1단계 이론값 2.40GB와 일치. 남은 VRAM은 ~7.6GB.

---
## 3️⃣ FP16 응답 저장 (나중에 4-bit와 비교용)

**이론 요약**
- 나중에 4-bit 응답과 **나란히 비교**하기 위해 FP16 응답을 먼저 기록
- EXAONE 4.0은 `apply_chat_template` 표준 방식 사용
- `temperature=0.1` → EXAONE 4.0 1.2B 한국어 권장값

### 미션 3-1. 추론 함수 (EXAONE 4.0 표준)

In [ ]:
def 질문하기(model, 질문, 최대토큰=150):
    # ✨ EXAONE 4.0 apply_chat_template 방식
    messages = [
        {"role": "user", "content": 질문}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,  # ✨ 필수!
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=최대토큰,
            temperature=0.1,   # ✨ EXAONE 4.0 1.2B 권장값
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    생성부분 = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(생성부분, skip_special_tokens=True).strip()

print("✅ 추론 함수 정의 완료")

### 미션 3-2. FP16 응답 생성 (법률 질문)

In [ ]:
테스트_질문 = "임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?"

응답_fp16 = 질문하기(model_fp16, 테스트_질문)
print(f"[FP16 응답]\n{응답_fp16}")

✅ **결과 확인**: 한국어 법률 답변이 생성됐나요?  
👉 이 응답을 **4-bit 응답과 비교**합니다.

### 미션 3-3. FP16 모델 메모리 해제

In [ ]:
del model_fp16
메모리_해제()
print_gpu_memory("해제 후")

✅ **결과 확인**: 0GB 근처로 내려갔나요?  
👉 `del + gc.collect() + empty_cache()` 3단계가 **GPU 메모리 해제 공식**.

---
## 4️⃣ BitsAndBytesConfig — 양자화 설정의 전부

**이론 요약**
- 4가지 설정으로 양자화 완전 제어
- `compute_dtype=bfloat16` — **EXAONE 4.0 공식 권장값**
- RTX 3080 (Ampere)은 bfloat16 네이티브 지원

```
EXAONE 4.0 공식 예제:
  torch_dtype="bfloat16"
  ↓
우리 양자화 설정:
  bnb_4bit_compute_dtype=torch.bfloat16  ← 동일 정책
```

💡 **왜 중요한가?**  
잘못된 `compute_dtype` 선택 시 학습 중 **NaN 발생**하거나 **느려집니다**.

### 미션 4-1. 환경에 맞는 compute_dtype 확인

In [ ]:
# RTX 3080은 Ampere 아키텍처 → bfloat16 네이티브 지원
지원여부 = torch.cuda.is_bf16_supported()
print(f"내 GPU bfloat16 지원: {지원여부}")

if 지원여부:
    compute_dtype_선택 = torch.bfloat16
    print(f"✅ 선택: bfloat16 (EXAONE 4.0 공식 권장)")
else:
    compute_dtype_선택 = torch.float16
    print(f"⚠️ bfloat16 미지원 → float16으로 대체")

### 미션 4-2. BitsAndBytesConfig 생성

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # 1️⃣ 4-bit 양자화 활성화
    bnb_4bit_quant_type="nf4",                  # 2️⃣ NF4 (정규분포 최적)
    bnb_4bit_compute_dtype=compute_dtype_선택,   # 3️⃣ EXAONE 4.0 권장 bfloat16
    bnb_4bit_use_double_quant=True,             # 4️⃣ Double Quantization
)

print("✅ 양자화 설정:")
print(f"  1️⃣ 4-bit 로드:     {bnb_config.load_in_4bit}")
print(f"  2️⃣ 양자화 타입:    {bnb_config.bnb_4bit_quant_type}")
print(f"  3️⃣ 계산 타입:      {bnb_config.bnb_4bit_compute_dtype}")
print(f"  4️⃣ Double Quant:  {bnb_config.bnb_4bit_use_double_quant}")

> 💡 이 `bnb_config`를 **Part 3(QLoRA 통합)에서 그대로 재사용**합니다.  
> EXAONE 4.0은 `bfloat16`이 공식 권장이므로 환경에 맞게 자동 선택됩니다.

---
## 5️⃣ 4-bit 양자화로 EXAONE 4.0 로드

**이론 요약**
- `quantization_config=bnb_config` **한 줄**이 양자화의 전부
- 이론값: 1.2B × 0.5byte = **0.6GB** (+Double Quant 오버헤드 ~0.4GB)
- 실측값: 약 1.0GB 예상

💡 **왜 중요한가?**  
FP16(2.4GB) → 4-bit(1.0GB) = **약 58% 추가 절감**.  
RTX 3080에 9GB 여유 → LoRA 학습 매우 여유.

### 미션 5-1. 4-bit로 모델 로드

In [ ]:
print("🔄 4-bit 로드 중...")

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,  # 🔑 양자화의 전부
    device_map="auto",
)
print_gpu_memory("4-bit 로드 후")

✅ **결과 확인**: 약 **1GB** 정도 사용되나요?  
👉 이론값보다 살짝 많을 수 있음 (Double Quant 오버헤드).  
❌ 2.4GB 이상이면 `bnb_config` 미적용 의심.

### 미션 5-2. FP16 vs 4-bit 비교표

In [ ]:
print("=" * 60)
print(f"{'방식':12s} {'VRAM':>8s} {'10GB 중':>10s} {'절감률':>10s}")
print("-" * 60)
print(f"{'FP16':12s} {'2.4GB':>8s} {'24%':>10s} {'기준':>10s}")
print(f"{'4-bit NF4':12s} {'~1.0GB':>8s} {'~10%':>10s} {'~58%':>10s}")
print("=" * 60)
print()
print("💡 EXAONE 4.0 1.2B + 4-bit → RTX 3080에 9GB 여유")

---
## 6️⃣ 양자화 전후 응답 품질 비교

**이론 요약**
- 메모리는 58% 줄었는데 **품질은 유지되는가?**
- 같은 질문 → 두 모델의 응답을 나란히 비교
- 법률 도메인이므로 **정확한 용어** 사용 여부 체크

💡 **왜 중요한가?**  
"양자화 = 품질 저하"라는 오해를 직접 깨뜨리는 순간.

### 미션 6-1. 4-bit 응답 생성

In [ ]:
응답_4bit = 질문하기(model_4bit, 테스트_질문)
print(f"[4-bit 응답]\n{응답_4bit}")

### 미션 6-2. FP16 vs 4-bit 나란히 비교

In [ ]:
print(f"[질문]\n{테스트_질문}")
print("\n" + "=" * 60)
print("[FP16 응답 - 2.4GB]")
print("=" * 60)
print(응답_fp16)
print("\n" + "=" * 60)
print("[4-bit 응답 - 1.0GB]")
print("=" * 60)
print(응답_4bit)

✅ **결과 확인**: 두 응답이 **비슷한 수준의 법률 정보**를 담고 있나요?  
👉 이것이 NF4 양자화의 힘입니다. **메모리 58% 절감 + 품질 거의 유지**.

---
## 7️⃣ 마무리 — Part 2(LoRA)로 연결

**이론 요약**
- 현재 상태: 4-bit EXAONE 4.0이 RTX 3080에 **1GB** 사용 중
- 남은 VRAM: 약 **9GB** → LoRA 학습에 매우 충분
- 하지만 양자화된 가중치는 **직접 학습 불가** → Part 2의 LoRA 필요

💡 **다음 파트에서 할 일**  
원본을 동결하고 **작은 어댑터만 학습**하는 LoRA 기법 학습.

### 미션 7-1. 현재 상태 요약

In [ ]:
print("=" * 55)
print("📋 Part 1 완료 상태")
print("=" * 55)
print(f"✅ 모델:       {MODEL_ID}")
print(f"✅ 양자화:     4-bit NF4 + Double Quantization")
print(f"✅ compute:    {bnb_config.bnb_4bit_compute_dtype}")
print(f"✅ 응답 품질:  FP16과 유사")
print()
print_gpu_memory("현재 상태")
print()
print("⏭️ Part 2 예고:")
print("   LoRA 어댑터로 '무엇을 학습할지' 결정")
print("⏭️ Part 3 예고:")
print("   양자화(Q) + LoRA = QLoRA로 법률 파인튜닝 시작!")

### 미션 7-2. 메모리 정리 (Part 2 실습 시작 전)

In [ ]:
del model_4bit
메모리_해제()
print_gpu_memory("최종 정리")

---

## 🎓 Part 1 정리

| 미션 | 배운 것 | EXAONE 4.0과의 연결 |
|---|---|---|
| 1 | 파라미터 × 바이트 공식 | 1.2B × 4byte = 4.8GB |
| 2 | FP16 실제 로드 → 2.4GB | 양자화 기준선 |
| 3 | 응답 저장 + 메모리 해제 | 반복 실험 패턴 |
| 4 | `BitsAndBytesConfig` 4가지 | Part 3에서 그대로 재사용 |
| 5 | 4-bit 실제 로드 → 1.0GB | QLoRA의 전제조건 |
| 6 | 품질 비교 | 양자화의 실용성 체감 |

### 🔑 EXAONE 4.0 핵심 포인트

- **HF 공식 지원**: `trust_remote_code` 불필요
- **공식 권장 dtype**: `bfloat16` (RTX 3080 Ampere에 최적)
- **작은 크기**: 1.2B → RTX 3080에 여유롭게 로드

### 🔜 다음 파트

**Part 2 — LoRA 기본 구조**  
`ΔW = A × B` 저랭크 분해 직접 구현 + EXAONE 4.0의 `target_modules` 이해 (Llama 계열 구조) + 작은 모델에서 어댑터 부착·저장·로드 연습

**Part 3 — QLoRA 통합 (양자화 + LoRA)**  
오늘 만든 `bnb_config` + Part 2의 `lora_config`를 결합 → EXAONE 4.0에 실전 적용 → Part 4 학습 실행 준비 완료